# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [23]:
# setup
from litellm import Cache, completion
import litellm
import gradio as gr

litellm.cache = Cache(type="local")

MODEL_LLAMA3_2 = "ollama/llama3.2"
MODEL_QWEN_2_5_CODER_14B = "ollama/qwen2.5-coder:14b"
MODEL_GPT_OSS_20B = "ollama/gpt-oss:20b"


In [24]:
# prepare prompts for testing connection

system_prompt = "You are an electrical engineer ready to answer math problems."
user_prompt = "what is the integral of 1/x ?"

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
    ]

In [ ]:
# Testing for connection to LLM using liteLLM
response = completion(model=MODEL_LLAMA3_2, messages=messages, caching=True)
response.choices[0].message.content

hidden = getattr(response, "_hidden_params", {}) or {}

print("Cache hit:", hidden.get("cache_hit"))
print("Metadata:", hidden)


print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

response2 = completion(model=MODEL_LLAMA3_2, messages=messages, caching=True)
response2.choices[0].message.content

hidden2 = getattr(response2, "_hidden_params", {}) or {}

print("Cache hit:", hidden2.get("cache_hit"))
print("Metadata:", hidden2)


print(f"Input tokens: {response2.usage.prompt_tokens}")
print(f"Output tokens: {response2.usage.completion_tokens}")
print(f"Total tokens: {response2.usage.total_tokens}")
print(f"Total cost: {response2._hidden_params["response_cost"]*100:.4f} cents")


Cache hit: None
Metadata: {'custom_llm_provider': 'ollama', 'region_name': None, 'optional_params': {}, 'litellm_call_id': '02931261-efb7-460b-8b4c-b006b860fb67', 'api_base': None, 'model_id': None, 'response_cost': 0.0, 'additional_headers': {}, 'litellm_model_name': 'ollama/llama3.2', '_response_ms': 8867.98}
Input tokens: 51
Output tokens: 82
Total tokens: 133
Total cost: 0.0000 cents
Cache hit: True
Metadata: {'additional_headers': {}, 'cache_hit': True, 'optional_params': {}, 'litellm_call_id': '48bf544e-bd31-4875-87c6-eec4e61093b2', 'api_base': None, 'model_id': None, 'response_cost': 0.0, 'litellm_model_name': 'ollama/llama3.2', '_response_ms': 0.533, 'cache_key': '5ab1ccda553d941bf53b5928cfa8d41bf2882b29bdfb73255042f71c7fadcb65'}
Input tokens: 51
Output tokens: 82
Total tokens: 133
Total cost: 0.0000 cents


In [ ]:
# Testing for connection to LLM using liteLLM
response3 = completion(model=MODEL_GPT_OSS_20B, messages=messages, caching=True)
response3.choices[0].message.content

hidden3 = getattr(response3, "_hidden_params", {}) or {}

print("Cache hit:", hidden3.get("cache_hit"))
print("Metadata:", hidden3)


print(f"Input tokens: {response3.usage.prompt_tokens}")
print(f"Output tokens: {response3.usage.completion_tokens}")
print(f"Total tokens: {response3.usage.total_tokens}")
print(f"Total cost: {response3._hidden_params["response_cost"]*100:.4f} cents")

response4 = completion(model=MODEL_GPT_OSS_20B, messages=messages, caching=True)
response4.choices[0].message.content

hidden4 = getattr(response4, "_hidden_params", {}) or {}

print("Cache hit:", hidden4.get("cache_hit"))
print("Metadata:", hidden4)


print(f"Input tokens: {response4.usage.prompt_tokens}")
print(f"Output tokens: {response4.usage.completion_tokens}")
print(f"Total tokens: {response4.usage.total_tokens}")
print(f"Total cost: {response4._hidden_params["response_cost"]*100:.4f} cents")


Cache hit: None
Metadata: {'custom_llm_provider': 'ollama', 'region_name': None, 'optional_params': {}, 'litellm_call_id': '5dc99dd1-a605-43e0-914c-2526c9c0ae41', 'api_base': None, 'model_id': None, 'response_cost': 0.0, 'additional_headers': {}, 'litellm_model_name': 'ollama/gpt-oss:20b', '_response_ms': 50055.541}
Input tokens: 93
Output tokens: 264
Total tokens: 357
Total cost: 0.0000 cents
Cache hit: True
Metadata: {'additional_headers': {}, 'cache_hit': True, 'optional_params': {}, 'litellm_call_id': '63801374-8736-4592-9382-732f2e545c7b', 'api_base': None, 'model_id': None, 'response_cost': 0.0, 'litellm_model_name': 'ollama/gpt-oss:20b', '_response_ms': 0.9990000000000001, 'cache_key': 'e908f5eec694624da7cb6f1dff4c4f06424cb059d8ebc5e7e8d36bc36e751331'}
Input tokens: 93
Output tokens: 264
Total tokens: 357
Total cost: 0.0000 cents


In [28]:

def call_model(model: str, messages: list[dict]) -> str:
    """Send the shared conversation to one local Ollama model."""

    response = completion(
        model=model,
        messages=messages,
        caching=True,
        temperature=0.7,
    )

    return response.choices[0].message.content

In [ ]:
def chat(message: str, history: list[dict]) -> str:
    """
    Gradio callback.

    Model A answers first.
    Model B then receives the user's message and Model A's answer.
    """

    shared_history = list(history)

    # Add the user's newest message.
    shared_history.append(
        {
            "role": "user",
            "content": message,
        }
    )

    # Model A responds to you.
    model_a_messages = [
        {
            "role": "system",
            "content": (
                "You are Model A in a group conversation with a human "
                "and another AI called Model B. Respond naturally and concisely."
            ),
        },
        *shared_history,
    ]

    model_a_reply = call_model(
        model=MODEL_GPT_OSS_20B,
        messages=model_a_messages,
    )

    # Model B sees both your message and Model A's new answer.
    model_b_messages = [
        {
            "role": "system",
            "content": (
                "You are Model B in a group conversation with a human "
                "and another AI called Model A. Consider Model A's response, "
                "then add your own useful perspective. Do not merely repeat it."
            ),
        },
        *shared_history,
        {
            "role": "assistant",
            "content": f"Model A said:\n{model_a_reply}",
        },
    ]

    model_b_reply = call_model(
        model=MODEL_QWEN_2_5_CODER_14B,
        messages=model_b_messages,
    )

    # ChatInterface displays this as one assistant turn.
    return (
        f"### Model A — `{MODEL_GPT_OSS_20B}`\n\n"
        f"{model_a_reply}\n\n"
        f"---\n\n"
        f"### Model B — `{MODEL_QWEN_2_5_CODER_14B}`\n\n"
        f"{model_b_reply}\n\n"
    )


In [41]:
latex_delimiters = [
    {
        "left": "$$",
        "right": "$$",
        "display": True,
    },
    {
        "left": "$",
        "right": "$",
        "display": False,
    },
    {
        "left": r"\[",
        "right": r"\]",
        "display": True,
    },
    {
        "left": r"\(",
        "right": r"\)",
        "display": False,
    },
]

chatbot = gr.Chatbot(
        type="messages",
        render_markdown=True,
        latex_delimiters=latex_delimiters,
        height=600,
)

In [42]:
demo = gr.ChatInterface(
    fn=chat,
    type="messages",
    title="Local Three-Way AI Chat",
    chatbot=chatbot
)

if __name__ == "__main__":
    demo.launch(
        inbrowser=True,
        inline=False,
    )

* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.
